# Chicago Sanitation Complaint Analysis

This notebook contains my contribution to the sanitation portion of the project. It cleans selected columns, creates charts, and summarizes key patterns.

## 1. Import libraries and load data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_excel(r"/mnt/data/311_Service_Requests_-_Sanitation_Code_Complaints_-_Historical_20260916.csv.xlsx")
print(df.shape)
print(df.columns.tolist())

## 2. Select the columns used in the analysis

In [ ]:
columns = [
  "Creation Date",
  "Street Address",
  "ZIP Code",
  "Ward",
  "Police District",
  "Community Area",
  "Latitude",
  "Longitude"
]
        sanitation = df[columns].copy()
        sanitation.head()

## 3. Clean the date column and create `Year`

In [ ]:
sanitation = sanitation.rename(columns={
    columns[0]: "Creation Date" if columns[0] != "Creation Date" else "Creation Date"
})
sanitation["Creation Date"] = pd.to_datetime(sanitation["Creation Date"], errors="coerce")
sanitation["Year"] = sanitation["Creation Date"].dt.year
sanitation[["Creation Date", "Year"]].head()

## 4. Check missing values

In [ ]:
# Standardize important column names if needed
rename_map = {}
if 'Creation Date' not in sanitation.columns:
    for c in sanitation.columns:
        if str(c).strip().lower() == 'creation date':
            rename_map[c] = 'Creation Date'
if 'Street Address' not in sanitation.columns:
    for c in sanitation.columns:
        if str(c).strip().lower() == 'street address':
            rename_map[c] = 'Street Address'
if 'ZIP Code' not in sanitation.columns:
    for c in sanitation.columns:
        if str(c).strip().lower() in ['zip code', 'zip']:
            rename_map[c] = 'ZIP Code'
if 'Ward' not in sanitation.columns:
    for c in sanitation.columns:
        if str(c).strip().lower() == 'ward':
            rename_map[c] = 'Ward'
if 'Police District' not in sanitation.columns:
    for c in sanitation.columns:
        if str(c).strip().lower() in ['police district', 'district']:
            rename_map[c] = 'Police District'
if 'Community Area' not in sanitation.columns:
    for c in sanitation.columns:
        if str(c).strip().lower() == 'community area':
            rename_map[c] = 'Community Area'
if 'Latitude' not in sanitation.columns:
    for c in sanitation.columns:
        if str(c).strip().lower() == 'latitude':
            rename_map[c] = 'Latitude'
if 'Longitude' not in sanitation.columns:
    for c in sanitation.columns:
        if str(c).strip().lower() == 'longitude':
            rename_map[c] = 'Longitude'

sanitation = sanitation.rename(columns=rename_map)

for col in ['ZIP Code', 'Ward', 'Police District', 'Community Area']:
    if col in sanitation.columns:
        sanitation[col] = sanitation[col].astype(str).str.strip()
        sanitation[col] = sanitation[col].replace({'nan': pd.NA, 'None': pd.NA, '': pd.NA, '<NA>': pd.NA})

if 'Street Address' in sanitation.columns:
    sanitation['Street Address'] = sanitation['Street Address'].astype(str).str.strip()
    sanitation['Street Address'] = sanitation['Street Address'].replace({'nan': pd.NA, '': pd.NA})

for col in ['Latitude', 'Longitude']:
    if col in sanitation.columns:
        sanitation[col] = pd.to_numeric(sanitation[col], errors='coerce')

sanitation.isna().sum().sort_values(ascending=False)

## 5. Complaints by Year

In [ ]:
year_counts = sanitation.dropna(subset=['Year']).groupby('Year').size().reset_index(name='Complaints')
year_counts['Year'] = year_counts['Year'].astype(int)

plt.figure(figsize=(8, 5))
plt.plot(year_counts['Year'], year_counts['Complaints'], marker='o')
plt.title('Chicago Sanitation Complaints by Year')
plt.xlabel('Year')
plt.ylabel('Number of Complaints')
plt.tight_layout()
plt.show()

year_counts.tail()

The year with the most recorded sanitation complaints in this dataset is 2016 with 21,139 complaints.

## 6. Top 10 ZIP Codes

In [ ]:
zip_counts = (
    sanitation.dropna(subset=['ZIP Code'])
    .groupby('ZIP Code')
    .size()
    .reset_index(name='Complaints')
    .sort_values('Complaints', ascending=False)
    .head(10)
)

plt.figure(figsize=(8, 5))
plt.barh(zip_counts['ZIP Code'].astype(str), zip_counts['Complaints'])
plt.title('Top 10 ZIP Codes by Sanitation Complaints')
plt.xlabel('Number of Complaints')
plt.ylabel('ZIP Code')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

zip_counts

The highest complaint total in this chart is for ZIP Code 60620.0 with 7,617 complaints.

## 7. Top 10 Community Areas

In [ ]:
community_counts = (
    sanitation.dropna(subset=['Community Area'])
    .groupby('Community Area')
    .size()
    .reset_index(name='Complaints')
    .sort_values('Complaints', ascending=False)
    .head(10)
)

plt.figure(figsize=(8, 5))
plt.barh(community_counts['Community Area'].astype(str), community_counts['Complaints'])
plt.title('Top 10 Community Areas by Sanitation Complaints')
plt.xlabel('Number of Complaints')
plt.ylabel('Community Area')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

community_counts

The highest complaint total in this chart is for Community Area 25.0 with 6,468 complaints.

## 8. Top 10 Wards

In [ ]:
ward_counts = (
    sanitation.dropna(subset=['Ward'])
    .groupby('Ward')
    .size()
    .reset_index(name='Complaints')
    .sort_values('Complaints', ascending=False)
    .head(10)
)

plt.figure(figsize=(8, 5))
plt.barh(ward_counts['Ward'].astype(str), ward_counts['Complaints'])
plt.title('Top 10 Wards by Sanitation Complaints')
plt.xlabel('Number of Complaints')
plt.ylabel('Ward')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

ward_counts

The highest complaint total in this chart is for Ward 6.0 with 6,431 complaints.

## 9. Geographic Distribution

In [ ]:
location_df = sanitation.dropna(subset=['Latitude', 'Longitude']).copy()
plot_df = location_df.sample(min(20000, len(location_df)), random_state=42) if len(location_df) > 20000 else location_df

plt.figure(figsize=(7, 7))
plt.scatter(plot_df['Longitude'], plot_df['Latitude'], s=2, alpha=0.35)
plt.title('Geographic Distribution of Sanitation Complaints')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.show()

The location plot shows that sanitation complaints are geographically clustered rather than evenly spread, which suggests hotspot areas across the city.

## 10. Key Findings

- The year with the most recorded sanitation complaints in this dataset is 2016 with 21,139 complaints.
- The highest complaint total in this chart is for ZIP Code 60620.0 with 7,617 complaints.
- The highest complaint total in this chart is for Community Area 25.0 with 6,468 complaints.
- The highest complaint total in this chart is for Ward 6.0 with 6,431 complaints.
- The location plot shows that sanitation complaints are geographically clustered rather than evenly spread, which suggests hotspot areas across the city.

### Limitations
- These are raw complaint counts, not complaint rates.
- Areas with more people may naturally have more complaints.
- Additional variables would be needed to explain why certain areas have more complaints.